# Preparación de datos y entrenamiento de modelo en SageMaker

En este notebook se implementa un flujo completo para el dataset del Titanic:

1. Lectura del dataset crudo.
2. Limpieza y feature engineering mediante un SageMaker Processing Job.
3. Generación de datasets de entrenamiento, validación y test.
4. Entrenamiento de un modelo con el algoritmo built-in XGBoost de SageMaker.

El objetivo del modelo es predecir la variable `Survived`.

## 1. Configuración inicial de SageMaker

Primero se configura la sesión de SageMaker, el bucket por defecto y el rol IAM que usará SageMaker para lanzar los jobs.

In [1]:
import os
import boto3
import sagemaker

from sagemaker import get_execution_role

sess = sagemaker.Session()
bucket = sess.default_bucket()
role = get_execution_role()
region = sess.boto_region_name

prefix = "sagemaker/titanic-lab"

print("Region:", region)
print("Bucket:", bucket)
print("Role:", role)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
Region: us-east-1
Bucket: sagemaker-us-east-1-173555533635
Role: arn:aws:iam::173555533635:role/LabRole


## 2. Comprobación del dataset crudo

El fichero de entrada será el CSV original del Titanic. Este fichero todavía contiene columnas sin limpiar, valores nulos y variables categóricas.

In [2]:
import pandas as pd

local_raw_data = "titanic/titanic-dataset.csv"

df = pd.read_csv(local_raw_data)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


## 3. Script de preprocesamiento externo

El preprocesamiento se encuentra en el archivo `processing_titanic.py`.

Este script será ejecutado por un SageMaker Processing Job. El notebook no contiene la lógica de limpieza directamente, sino que invoca ese script externo, igual que en los ejemplos de clase.

In [4]:
import os

script_path = "processing_titanic.py"

print("Ruta absoluta:", os.path.abspath(script_path))
print("Existe:", os.path.exists(script_path))

Ruta absoluta: /home/ec2-user/SageMaker/processing_titanic.py
Existe: True


## 4. Lanzamiento del Processing Job

Se crea un `SKLearnProcessor` para ejecutar el script de preprocesamiento en SageMaker.

El job recibe como entrada el dataset crudo del Titanic y genera tres salidas:

- entrenamiento
- validación
- test

In [5]:
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

sklearn_processor = SKLearnProcessor(
    framework_version="1.2-1",
    role=role,
    instance_type="ml.m5.xlarge",
    instance_count=1,
    base_job_name="titanic-processing"
)

sklearn_processor.run(
    code="processing_titanic.py",
    inputs=[
        ProcessingInput(
            source=local_raw_data,
            destination="/opt/ml/processing/input"
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            source="/opt/ml/processing/train",
            destination=f"s3://{bucket}/{prefix}/train"
        ),
        ProcessingOutput(
            output_name="validation",
            source="/opt/ml/processing/validation",
            destination=f"s3://{bucket}/{prefix}/validation"
        ),
        ProcessingOutput(
            output_name="test",
            source="/opt/ml/processing/test",
            destination=f"s3://{bucket}/{prefix}/test"
        )
    ]
)

INFO:sagemaker:Creating processing-job with name titanic-processing-2026-05-30-10-05-40-099


............Reading raw data from: /opt/ml/processing/input/titanic-dataset.csv
Original shape: (891, 12)
Processed shape: (891, 17)
Processed columns: ['Survived', 'Pclass', 'Age', 'Fare', 'Sex_female', 'Sex_male', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare', 'Family_size_Alone', 'Family_size_Large', 'Family_size_Small']
Train shape: (623, 17)
Validation shape: (134, 17)
Test shape: (134, 17)
Processing completed successfully



## 5. Rutas S3 generadas

El Processing Job ha dejado los datasets procesados en S3. Estas rutas se usarán como entrada del entrenamiento.

In [6]:
train_s3_uri = f"s3://{bucket}/{prefix}/train"
validation_s3_uri = f"s3://{bucket}/{prefix}/validation"
test_s3_uri = f"s3://{bucket}/{prefix}/test"

train_file_s3_uri = f"{train_s3_uri}/train.csv"
validation_file_s3_uri = f"{validation_s3_uri}/validation.csv"
test_file_s3_uri = f"{test_s3_uri}/test.csv"

print("Train:", train_file_s3_uri)
print("Validation:", validation_file_s3_uri)
print("Test:", test_file_s3_uri)

Train: s3://sagemaker-us-east-1-173555533635/sagemaker/titanic-lab/train/train.csv
Validation: s3://sagemaker-us-east-1-173555533635/sagemaker/titanic-lab/validation/validation.csv
Test: s3://sagemaker-us-east-1-173555533635/sagemaker/titanic-lab/test/test.csv


## 6. Entrenamiento con XGBoost built-in

Se usará el algoritmo built-in XGBoost de SageMaker.

Como el problema es de clasificación binaria, se configura el objetivo `binary:logistic`.

El dataset ya está en formato CSV sin cabecera y con `Survived` como primera columna.

In [7]:
from sagemaker.inputs import TrainingInput

xgboost_container = sagemaker.image_uris.retrieve(
    "xgboost",
    region,
    "1.7-1"
)

print(xgboost_container)

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1


In [8]:
s3_input_train = TrainingInput(
    s3_data=train_s3_uri,
    content_type="csv"
)

s3_input_validation = TrainingInput(
    s3_data=validation_s3_uri,
    content_type="csv"
)

In [9]:
xgb = sagemaker.estimator.Estimator(
    image_uri=xgboost_container,
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/model-output",
    sagemaker_session=sess,
    base_job_name="titanic-xgboost"
)

xgb.set_hyperparameters(
    objective="binary:logistic",
    eval_metric="auc",
    max_depth=4,
    eta=0.2,
    gamma=2,
    min_child_weight=1,
    subsample=0.8,
    num_round=100
)

INFO:botocore.credentials:Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole


## 7. Ejecución del Training Job

Se lanza el entrenamiento usando los canales `train` y `validation`.

En los logs del entrenamiento aparecerá la métrica `validation:auc`.

In [10]:
xgb.fit(
    {
        "train": s3_input_train,
        "validation": s3_input_validation
    }
)

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: titanic-xgboost-2026-05-30-10-08-33-003


2026-05-30 10:08:34 Starting - Starting the training job...
2026-05-30 10:08:48 Starting - Preparing the instances for training...
2026-05-30 10:09:14 Downloading - Downloading input data...
2026-05-30 10:09:39 Downloading - Downloading the training image...
2026-05-30 10:10:30 Training - Training image download completed. Training in progress../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-05-30 10:10:37.702 ip-10-0-251-207.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-05-30 10:10:37.778 ip-10-0-251-207.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-05-30:10:10:38:INFO] Imported framework sagemaker_xgboost_container.training

## 8. Artefacto del modelo entrenado

Cuando termina el entrenamiento, SageMaker guarda el modelo en S3 como un artefacto `model.tar.gz`.

In [11]:
model_artifact = xgb.model_data
print("Modelo guardado en:", model_artifact)

Modelo guardado en: s3://sagemaker-us-east-1-173555533635/sagemaker/titanic-lab/model-output/titanic-xgboost-2026-05-30-10-08-33-003/output/model.tar.gz


## 9. Despliegue temporal del modelo

Esta parte es opcional. Se despliega un endpoint temporal para hacer predicciones.

Después de probarlo, se debe eliminar el endpoint para evitar costes.

In [12]:
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import StringDeserializer

predictor = xgb.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    serializer=CSVSerializer(),
    deserializer=StringDeserializer()
)

INFO:sagemaker:Creating model with name: titanic-xgboost-2026-05-30-10-11-20-176
INFO:sagemaker:Creating endpoint-config with name titanic-xgboost-2026-05-30-10-11-20-176
INFO:sagemaker:Creating endpoint with name titanic-xgboost-2026-05-30-10-11-20-176


------!

## 10. Descarga del dataset de test

Se descarga el conjunto de test procesado para probar algunas predicciones.

In [13]:
test_local_path = "test.csv"

s3_client = boto3.client("s3")
test_key = f"{prefix}/test/test.csv"

s3_client.download_file(bucket, test_key, test_local_path)

test_data = pd.read_csv(test_local_path, header=None)
test_data.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
0,1,3,29,7.8958,0,1,1,0,0,0,0,1,0,0,1,0,0
1,1,1,27,30.5000,0,1,0,0,1,0,0,1,0,0,1,0,0
2,1,3,24,15.8500,1,0,0,0,1,0,0,0,1,0,0,0,1
3,0,2,28,10.5000,0,1,0,0,1,0,0,1,0,0,1,0,0
4,1,1,39,83.1583,1,0,1,0,0,0,1,0,0,0,0,0,1


In [14]:
import numpy as np

y_test = test_data.iloc[:, 0]
X_test = test_data.iloc[:, 1:]

sample = X_test.head(10)

raw_predictions = predictor.predict(sample.to_numpy())
raw_predictions

'{"predictions": [{"score": 0.1837504655122757}, {"score": 0.4979955852031708}, {"score": 0.6193532347679138}, {"score": 0.09846007078886032}, {"score": 0.9730278253555298}, {"score": 0.9016890525817871}, {"score": 0.06536893546581268}, {"score": 0.0753103494644165}, {"score": 0.8879069089889526}, {"score": 0.06176874786615372}]}'

In [16]:
import json
import numpy as np
import pandas as pd

# Convertir el texto JSON en diccionario de Python
predictions_json = json.loads(raw_predictions)

# Extraer los scores
predicted_probabilities = np.array(
    [item["score"] for item in predictions_json["predictions"]]
)

# Convertir probabilidades en clases
predicted_classes = (predicted_probabilities >= 0.5).astype(int)

# Crear tabla
pd.DataFrame({
    "real": y_test.head(len(predicted_probabilities)).values,
    "probabilidad_supervivencia": predicted_probabilities,
    "prediccion": predicted_classes
})

,real,probabilidad_supervivencia,prediccion
0,1,0.183750,0
1,1,0.497996,0
2,1,0.619353,1
3,0,0.098460,0
4,1,0.973028,1
5,0,0.901689,1
6,0,0.065369,0
7,0,0.075310,0
8,1,0.887907,1
9,0,0.061769,0


## 11. Eliminación del endpoint

Se elimina el endpoint para evitar costes innecesarios.

In [17]:
predictor.delete_endpoint()

INFO:sagemaker:Deleting endpoint configuration with name: titanic-xgboost-2026-05-30-10-11-20-176
INFO:sagemaker:Deleting endpoint with name: titanic-xgboost-2026-05-30-10-11-20-176
